# Fix hs_cty_fips Codes

Two fixes:
1. **Public schools**: zero-pad 4-digit FIPS codes to 5 digits
2. **Private schools**: look up FIPS from PSS 2019-20 data (PSTANSI + PCNTY)

In [1]:
import pandas as pd

df = pd.read_csv('../data/final_data_v3.csv', dtype={'hs_cty_fips': str})
pss = pd.read_csv('../data/pss_2019-20.csv', low_memory=False, dtype={'PPIN': str, 'PSTANSI': str, 'PCNTY': str})

print('Rows in final_data_v3:', len(df))
print('Public rows:', (df['school_type'] == 'public').sum())
print('Private rows:', (df['school_type'] == 'private').sum())

Rows in final_data_v3: 1048575
Public rows: 803825
Private rows: 244750


In [2]:
# Inspect current hs_cty_fips values
print('Sample public hs_cty_fips:')
print(df[df['school_type'] == 'public']['hs_cty_fips'].head(10).tolist())
print()
print('Sample private hs_cty_fips:')
print(df[df['school_type'] == 'private']['hs_cty_fips'].head(10).tolist())

Sample public hs_cty_fips:
['45035.0', '36061.0', '39153.0', '20021.0', '32003.0', '36055.0', '13139.0', '49035.0', nan, '40145.0']

Sample private hs_cty_fips:
['10101.0', '89.0', '19.0', '73.0', '201.0', '912.0', '6136.0', '71051.0', '63.0', '95.0']


## Fix 1: Public schools — zero-pad 4-digit FIPS to 5 digits

In [3]:
def fix_public_fips(val):
    """Convert float-string like '1555.0' or '45035.0' to zero-padded 5-digit string."""
    if pd.isna(val) or val == 'nan':
        return val
    # Remove decimal part (stored as float in CSV)
    try:
        return str(int(float(val))).zfill(5)
    except (ValueError, TypeError):
        return val

public_mask = df['school_type'] == 'public'
df.loc[public_mask, 'hs_cty_fips'] = df.loc[public_mask, 'hs_cty_fips'].apply(fix_public_fips)

print('Sample public hs_cty_fips after fix:')
print(df[public_mask]['hs_cty_fips'].head(10).tolist())

# Check for any remaining 4-digit codes
four_digit = df[public_mask]['hs_cty_fips'].dropna().apply(lambda x: len(str(x)) < 5 if x != 'nan' else False)
print(f'\nPublic rows still with <5 digit FIPS: {four_digit.sum()}')

Sample public hs_cty_fips after fix:
['45035', '36061', '39153', '20021', '32003', '36055', '13139', '49035', nan, '40145']

Public rows still with <5 digit FIPS: 0


## Fix 2: Private schools — look up FIPS from PSS 2019-20 (PSTANSI + PCNTY)

In [4]:
# Build FIPS lookup from PSS: PSTANSI zero-padded to 2 digits + PCNTY zero-padded to 3 digits
pss['pss_fips'] = pss['PSTANSI'].str.zfill(2) + pss['PCNTY'].str.zfill(3)
pss_lookup = pss[['PPIN', 'pss_fips']].drop_duplicates('PPIN').set_index('PPIN')

print('PSS lookup sample:')
print(pss_lookup.head(5))
print(f'\nTotal unique PPINs in PSS: {len(pss_lookup)}')

PSS lookup sample:
         pss_fips
PPIN             
A1992166    01073
BB200010    01015
A1900009    01043
00001026    01067
A1500002    01051

Total unique PPINs in PSS: 21572


/var/folders/wz/pjx159kj141cgdkvkdrlfb040000gn/T/ipykernel_30058/1591327135.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pss['pss_fips'] = pss['PSTANSI'].str.zfill(2) + pss['PCNTY'].str.zfill(3)


In [5]:
private_mask = df['school_type'] == 'private'
private_hs_ids = df.loc[private_mask, 'hs_id'].astype(str)

# Check match rate before applying
matched = private_hs_ids.isin(pss_lookup.index)
print(f'Private rows: {private_mask.sum()}')
print(f'Matched in PSS: {matched.sum()}')
print(f'Not matched: {(~matched).sum()}')
print()
print('Unmatched hs_id samples:')
print(df.loc[private_mask & ~matched, 'hs_id'].head(10).tolist())

Private rows: 244750
Matched in PSS: 244750
Not matched: 0

Unmatched hs_id samples:
[]


In [6]:
# Apply PSS FIPS to private schools
df.loc[private_mask, 'hs_cty_fips'] = df.loc[private_mask, 'hs_id'].astype(str).map(pss_lookup['pss_fips'])

print('Sample private hs_cty_fips after fix:')
print(df[private_mask]['hs_cty_fips'].head(10).tolist())

Sample private hs_cty_fips after fix:
['01101', '17089', '22019', '06073', '48201', '12009', '36061', '51710', '22063', '22095']


## Validation

In [7]:
print('=== FIPS length distribution ===')
fips_lengths = df['hs_cty_fips'].dropna().apply(lambda x: len(str(x)) if str(x) != 'nan' else None).value_counts().sort_index()
print(fips_lengths)

print('\n=== Public school FIPS length distribution ===')
print(df[public_mask]['hs_cty_fips'].dropna().apply(lambda x: len(str(x))).value_counts().sort_index())

print('\n=== Private school FIPS length distribution ===')
print(df[private_mask]['hs_cty_fips'].dropna().apply(lambda x: len(str(x))).value_counts().sort_index())

print('\n=== Null hs_cty_fips by school type ===')
print(df.groupby('school_type')['hs_cty_fips'].apply(lambda x: x.isna().sum()))

=== FIPS length distribution ===
hs_cty_fips
5    1026823
Name: count, dtype: int64

=== Public school FIPS length distribution ===
hs_cty_fips
5    782073
Name: count, dtype: int64

=== Private school FIPS length distribution ===
hs_cty_fips
5    244750
Name: count, dtype: int64

=== Null hs_cty_fips by school type ===
school_type
private        0
public     21752
Name: hs_cty_fips, dtype: int64


## Save output

In [8]:
df.to_csv('../data/final_data_v4.csv', index=False)
print('Saved to ../data/final_data_v4.csv')
print(f'Total rows: {len(df)}')

Saved to ../data/final_data_v4.csv
Total rows: 1048575
